# Sheep vision → dataset: one model for BOTH fleeing and foraging

**Goal.** Prove that a single small CNN+FNN can learn, from **only** what a sheep perceives,
the `(dx, dy)` heading it should move — the same quantity `RuleBrain` produces — across the
sheep's two entity-driven behaviours **at once**:

* **flee** a nearby fox (move *away* from the threat), and
* **forage** toward grass (move *toward* where grass is densest).

We reuse the real simulation code unchanged (`sim/`): build a world, drop in **one sheep**
(the observer) plus, in flee scenes, **one nearby fox**, and run the real **perception →
`RuleBrain`** pipeline. This is the sheep counterpart of `notebooks/vision/fox/`, but instead
of two isolated models it is **one dataset + one model** that must learn the sheep's
*arbitration*: with a fox close the sheep flees even while hungry over grass; with no fox the
hungry sheep forages.

**Channels (chosen for this task).** The model reads **3** egocentric grid channels —
`terrain` (context), `food` (the grass field), and `threat` (the fox blip) — and outputs the
unit `(dx, dy)` heading. Because every scene holds hunger constant, the flee-vs-forage
decision is explained by the **threat channel alone** (fox present → flee; empty → forage),
so the mapping from vision to heading is well-posed. Nothing in `sim/` is modified.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make the ecosystem package importable no matter which directory the kernel started in.
_here = Path.cwd()
REPO = next((c for c in [_here, *_here.parents]
             if (c / "config.py").exists() and (c / "sim").is_dir()), _here)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from config import make_config, SHEEP, FOX
from sim.world import World
from sim.environment import Environment
from sim.entities import Entities
from sim.grid import SpatialGrid
from sim.perception import Perception, SH_TERRAIN, SH_FOOD, SH_THREAT
from sim.brain import RuleBrain, A_DX, A_DY, _FLEE_TRIGGER
from sim.systems import vegetation
from sim import genome as gn

print("repo:", REPO)

## 1. Parameters and world

The perception window is fixed by the sim at `K = 2*R+1`, but a sheep only sees within its own
`sensory_range`. We pin the sheep's eye radius to **12 cells** (gene range is `[8, 22]`) and
crop the stored window to ±12 (a `25×25` view, identical to the fox experiment), which is
loss-less: everything outside the eye disc is zero anyway.

* **Flee** scenes place a fox at `dist ∈ [2, 0.40×sensory]`, safely inside `RuleBrain`'s
  `_FLEE_TRIGGER = 0.45×sensory`, so the flee branch (priority 1) fires and returns a vector
  *away* from the fox.
* **Forage** scenes place no fox; the true heading is the direction to the grass
  **centre-of-mass** in view (a smooth, well-posed "toward grass" heading — see §4).

In [ ]:
# --- experiment parameters -------------------------------------------------
WORLD_SEED    = 12345      # fixes terrain + rivers (identical map every run)
SHEEP_SENSORY = 12.0       # sheep eye radius in cells (gene range [8, 22]); pinned here
CROP_R        = 12         # crop the egocentric window to +/- this many cells
WIN           = 2 * CROP_R + 1                    # -> 25 x 25
N_SCENARIOS   = 1000       # each -> 1 true + several diverse false rows
GEN_SEED      = 20240709   # reproducible scenario sampling
FLEE_FRACTION = 0.5        # target share of flee scenes (rest forage)
FOX_MAX_FRAC  = 0.40       # place the fox within this fraction of sensory (< _FLEE_TRIGGER)
MIN_COM_D     = 1.5        # forage: grass centre-of-mass must be this far off-centre (cells)

POS_TOL      = np.radians(20.0)   # within this of the true heading -> label 1
NEG_MARGIN   = np.radians(45.0)   # beyond this from the true heading -> label 0
N_POS_JITTER = 1                  # extra near-true positives per scene (smooths the target)

DATA_PATH = REPO / "notebooks" / "vision" / "sheep" / "sheep_vision_dataset.csv"

# --- build the headless world + the pieces perception needs ----------------
cfg   = make_config(world_seed=WORLD_SEED, seed=7)
world = World(cfg.world)
env   = Environment(cfg.env, np.random.default_rng(1))
ent   = Entities(cfg)
veg   = vegetation.initial_field(world, np.random.default_rng(2))   # sheep-food field
temp_field = env.temperature_field(world.static_temp)               # env frozen at t=0

# one spatial hash per species (perception queries them); rebuilt every scenario
grids = {SHEEP: SpatialGrid(world.w, world.h, cfg.sim.grid_cell_size),
         FOX:   SpatialGrid(world.w, world.h, cfg.sim.grid_cell_size)}
perc  = Perception(cfg, world, ent, env)
brain = RuleBrain(np.random.default_rng(3))

# candidate sheep/fox cells: passable, dry, and NOT forest cover (mirrors the fox notebook).
land_mask = world.passable & ~world.water_any & ~world.cover
land_ys, land_xs = np.nonzero(land_mask)

sheep_spec, fox_spec = cfg.species[SHEEP], cfg.species[FOX]
S_IDX = gn.GENE_INDEX["sensory_range"]

# egocentric cell-offset stencils for the cropped WIN x WIN window (centre = the sheep)
_offs = np.arange(WIN) - CROP_R
_OY, _OX = np.meshgrid(_offs, _offs, indexing="ij")
_OX = _OX.astype(np.float32); _OY = _OY.astype(np.float32)
print(f"perception window K={perc.K}; stored crop {WIN}x{WIN}; land cells={land_xs.size}; "
      f"flee trigger={_FLEE_TRIGGER}")

## 2. One scenario = one perception → true heading

`make_scenario` clears the world, places a hungry sheep (and a nearby fox in flee scenes),
and runs the real perception + brain. Details mirrored from the sim:

* **The sheep is made hungry** (`hunger=0.75`) so a food drive is always latent — in flee
  scenes this forces `RuleBrain` to let flee **override** foraging, exactly the arbitration we
  want the model to learn.
* **Grids are rebuilt and wired exactly as `Simulation.step` does**, so the sheep's `threat`
  channel really contains the fox and its `food` channel really contains the grass field.
* **Targets.** Flee: the `RuleBrain` flee vector (away from the nearest fox). Forage: the
  direction to the grass **centre-of-mass** in view. (The `RuleBrain`'s exact best-grass
  *cell* is an ill-posed argmax of a noisy field — near-tied cells flip the "correct"
  direction — so we regress the smooth centroid the same perceived food channel implies; it
  captures "move toward where the grass is" and is cleanly learnable.)

In [ ]:
def rebuild_grids():
    """Re-index both species into their spatial hashes (mirrors Simulation._rebuild_grids)."""
    for sid, g in grids.items():
        sidx = np.nonzero(ent.species_mask(sid))[0]
        g.rebuild(sidx, ent.pos_x, ent.pos_y)


def make_scenario(rng, flee):
    """Place a hungry sheep (+ a close fox if `flee`); run perception -> RuleBrain.

    Returns (terrain, food, threat, dx, dy, meta), or None if no valid scene was found.
      terrain, food, threat : (WIN, WIN) cropped sheep channels
      dx, dy                : the unit TRUE heading (away from the fox / toward grass mass)
    """
    ent.kill(ent.alive_indices())                       # fresh, empty world

    fxp = fyp = None
    for _ in range(200):                                # sample a valid layout
        k = int(rng.integers(0, land_xs.size))
        sx0, sy0 = land_xs[k] + 0.5, land_ys[k] + 0.5   # sheep (observer) position
        if flee:
            dist = rng.uniform(2.0, FOX_MAX_FRAC * SHEEP_SENSORY)   # inside the flee trigger
            ang  = rng.uniform(0.0, 2 * np.pi)
            fxp, fyp = sx0 + dist * np.cos(ang), sy0 + dist * np.sin(ang)
            fcx, fcy = int(fxp), int(fyp)
            if not (0 <= fcx < world.w and 0 <= fcy < world.h):
                continue
            if world.water_any[fcy, fcx]:
                continue
        break
    else:
        return None

    sg = gn.random_genomes(sheep_spec, 1, rng)
    sg[0, S_IDX] = SHEEP_SENSORY
    sheep_slot = int(ent.spawn(sheep_spec, sg, np.array([[sx0, sy0]], np.float32), rng,
                               energy=0.5, age=np.float32(sheep_spec.maturity_age * 2))[0])
    ent.hunger[sheep_slot] = 0.75      # hungry -> a food drive is always latent
    ent.thirst[sheep_slot] = 0.05
    ent.energy[sheep_slot] = 0.50

    if flee:
        fg = gn.random_genomes(fox_spec, 1, rng)
        ent.spawn(fox_spec, fg, np.array([[fxp, fyp]], np.float32), rng,
                  energy=0.6, age=np.float32(fox_spec.maturity_age * 2))

    rebuild_grids()
    perc._species_grids = grids
    perc.veg = veg
    perc.temp_field = temp_field
    obs_by_species, idx = perc.build()
    act = brain.decide(obs_by_species, idx)

    sobs = obs_by_species[SHEEP]
    sgrids = sobs.grids[0]                              # (5, K, K); the sheep is the only row
    R = perc.R
    sl = slice(R - CROP_R, R + CROP_R + 1)
    terrain = sgrids[SH_TERRAIN, sl, sl].astype(np.float32).copy()
    food    = sgrids[SH_FOOD,    sl, sl].astype(np.float32).copy()
    threat  = sgrids[SH_THREAT,  sl, sl].astype(np.float32).copy()

    if flee:
        # target = the sheep RuleBrain's flee vector (away from the nearest fox)
        grow = int(np.searchsorted(idx, sheep_slot))
        dx, dy = float(act[grow, A_DX]), float(act[grow, A_DY])
        if (dx == 0.0 and dy == 0.0) or threat.sum() == 0:
            return None                                # sheep didn't perceive / react to a fox
        fox_dx, fox_dy = fxp - sx0, fyp - sy0          # offset TO the fox
        away = np.array([-fox_dx, -fox_dy], np.float32)
        away /= np.hypot(*away) + 1e-9
        if dx * away[0] + dy * away[1] < 0.9:          # RuleBrain must actually flee
            return None
        dist = float(np.hypot(fox_dx, fox_dy))
    else:
        # target = direction to the grass CENTRE-OF-MASS in view (well-posed "toward grass")
        total = float(food.sum())
        if total < 1e-6:
            return None
        cxo = float((food * _OX).sum() / total)
        cyo = float((food * _OY).sum() / total)
        dist = float(np.hypot(cxo, cyo))
        if dist < MIN_COM_D:                           # need a clear directional signal
            return None
        dx, dy = cxo / dist, cyo / dist

    meta = dict(mode=1 if flee else 0, dx=dx, dy=dy, dist=dist,
                threat_cells=int((threat > 0).sum()))
    return terrain, food, threat, dx, dy, meta

## 3. Sanity check + visualization

Draw one **flee** scene and one **forage** scene. In the flee row the red arrow (the true
heading) points *away* from the fox blip in the threat panel; in the forage row it points
*toward* the brighter grass in the food panel.

In [ ]:
rng_viz = np.random.default_rng(0)
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for r, flee in enumerate([True, False]):
    out = None
    while out is None:
        out = make_scenario(rng_viz, flee)
    terrain, food, threat, dx, dy, meta = out
    c = CROP_R
    panels = [(terrain, "terrain", "viridis"), (food, "food (grass)", "YlGn"),
              (threat, "threat (fox)", "magma")]
    for a, (grid, title, cmap) in zip(axes[r], panels):
        a.imshow(grid, origin="upper", cmap=cmap)
        a.plot(c, c, "wo", ms=8, mec="k")
        a.arrow(c, c, dx * 5, dy * 5, color="red", width=0.25, head_width=1.2,
                length_includes_head=True)
        a.set_title(title); a.set_xticks([]); a.set_yticks([])
    axes[r][0].set_ylabel("FLEE" if flee else "FORAGE", fontsize=13)
    print(f"{'FLEE  ' if flee else 'FORAGE'} true heading (dx,dy)=({dx:+.2f},{dy:+.2f}) "
          f"dist={meta['dist']:.1f} threat_cells={meta['threat_cells']}")
fig.suptitle("Sheep egocentric view + TRUE heading (red arrow): away from fox / toward grass")
plt.tight_layout(); plt.show()

## 4. Generate the dataset (true + diverse false headings)

For each scene we emit one row per candidate heading, all sharing the **same** perception,
exactly like the fox notebook:

| heading | offset from true | label |
|:--|:--|:--:|
| the true heading (+ a small jitter) | `≤ POS_TOL` | **1** |
| sideways | ≈ ±90° | 0 |
| oblique | ≈ ±135° | 0 |
| opposite | 180° | 0 |
| random off-angle | `≥ NEG_MARGIN` | 0 |

We draw whichever mode is still under target so the dataset stays balanced (forage scenes are
filtered more often, so a fixed coin would skew toward flee). We store `cos` = cosine between
the row's heading and the true heading (≈ +1 for true, ≈ 0 sideways, ≈ −1 opposite).

In [ ]:
def candidate_headings(true_ang, rng):
    """(angle, label) pairs: the true heading (+jitter) and a spread of wrong ones."""
    cands = [(true_ang, 1)]
    for _ in range(N_POS_JITTER):
        cands.append((true_ang + rng.uniform(-POS_TOL, POS_TOL), 1))
    s = 1.0 if rng.random() < 0.5 else -1.0                    # random left/right per scene
    offsets = [s * np.radians(90), -s * np.radians(135), np.radians(180),   # side/oblique/opp
               s * rng.uniform(NEG_MARGIN, np.radians(180))]               # random off-angle
    cands += [(true_ang + off, 0) for off in offsets]
    return cands


rng = np.random.default_rng(GEN_SEED)
t_rows, f_rows, x_rows, meta_rows = [], [], [], []
made = attempts = n_flee = 0
n_flee_target = int(round(N_SCENARIOS * FLEE_FRACTION))
n_forage_target = N_SCENARIOS - n_flee_target
while made < N_SCENARIOS:
    attempts += 1
    need_flee = n_flee < n_flee_target
    need_forage = (made - n_flee) < n_forage_target
    flee = (rng.random() < FLEE_FRACTION) if (need_flee and need_forage) else need_flee
    out = make_scenario(rng, flee)
    if out is None:
        continue
    terrain, food, threat, dx, dy, meta = out
    true_ang = np.arctan2(dy, dx)
    twx, twy = np.cos(true_ang), np.sin(true_ang)
    tf, ff, xf = terrain.ravel(), food.ravel(), threat.ravel()
    for ang, label in candidate_headings(true_ang, rng):
        cdx, cdy = float(np.cos(ang)), float(np.sin(ang))
        t_rows.append(tf); f_rows.append(ff); x_rows.append(xf)
        meta_rows.append((made, meta["mode"], label, cdx, cdy, dx, dy, meta["dist"],
                          cdx * twx + cdy * twy))     # last col = cos vs the true heading
    n_flee += meta["mode"]
    made += 1

t_arr = np.asarray(t_rows, np.float32)
f_arr = np.asarray(f_rows, np.float32)
x_arr = np.asarray(x_rows, np.float32)
meta_arr = np.asarray(meta_rows, np.float32)
print(f"{made} scenarios ({attempts} attempts) -> {len(t_rows)} rows; window {WIN}x{WIN}")
print(f"mode balance: flee={n_flee} forage={made - n_flee}")
print("label balance:", dict(zip(*np.unique(meta_arr[:, 2].astype(int), return_counts=True))))

cols_meta = ["scenario_id", "mode", "label", "dx", "dy", "tgt_dx", "tgt_dy", "dist", "cos"]
cols_t = [f"t_{i}" for i in range(WIN * WIN)]
cols_f = [f"f_{i}" for i in range(WIN * WIN)]
cols_x = [f"x_{i}" for i in range(WIN * WIN)]
df = pd.concat([
    pd.DataFrame(meta_arr, columns=cols_meta),
    pd.DataFrame(t_arr, columns=cols_t),
    pd.DataFrame(f_arr, columns=cols_f),
    pd.DataFrame(x_arr, columns=cols_x),
], axis=1)
for col in ("scenario_id", "mode", "label"):
    df[col] = df[col].astype(int)

DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(DATA_PATH, index=False, float_format="%.5f")
print("saved:", DATA_PATH, f"({DATA_PATH.stat().st_size/1e6:.1f} MB)")
df[["scenario_id", "mode", "label", "dx", "dy", "cos", "dist"]].head(8)

## 5. What's in the CSV

Columns:

* `scenario_id` — groups all candidate headings that share one perception (split on this).
* `mode` — `1` = flee (a fox is in view), `0` = forage (no fox).
* `label` — `1` = the true heading (+jitter), `0` = a wrong heading (sideways / oblique / …).
* `dx, dy` — the candidate heading for this row (unit vector).
* `tgt_dx, tgt_dy` — the **true** heading (away from fox / toward grass); the regression target.
* `dist` — distance to the salient entity (the fox, or the grass centre-of-mass), in cells.
* `cos` — cosine between this row's heading and the true heading (graded goodness).
* `t_0…` / `f_0…` / `x_0…` — flattened **terrain** / **food** / **threat** channels
  (`arr.reshape(WIN, WIN)`).

Next: **`train_model.ipynb`** trains one CNN+FNN direction regressor on `tgt_dx, tgt_dy` and
recovers the heading for both modes. Use **`show_dataset.py`** to flip through the rows and see
each heading over the three channels.